In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

comment_category_prediction_challenge_path = kagglehub.competition_download('comment-category-prediction-challenge')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ===================== IMPORTS =====================
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from scipy.sparse import hstack

# ===================== LOAD =====================
train = pd.read_csv('/kaggle/input/competitions/comment-category-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/comment-category-prediction-challenge/test.csv')

# ===================== CLEAN =====================
train['comment'] = train['comment'].fillna('').astype(str)
test['comment'] = test['comment'].fillna('').astype(str)

binary_cols = ['race', 'religion', 'gender', 'disability']

for col in binary_cols:
    train[col] = train[col].replace('none', np.nan)
    test[col] = test[col].replace('none', np.nan)

    train[col] = train[col].map({True:1, False:0})
    test[col] = test[col].map({True:1, False:0})

    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

# ===================== FEATURE ENGINEERING =====================
train['created_date'] = pd.to_datetime(train['created_date'])
test['created_date'] = pd.to_datetime(test['created_date'])

for df in [train, test]:
    df['year'] = df['created_date'].dt.year
    df['month'] = df['created_date'].dt.month
    df['day'] = df['created_date'].dt.day
    df['hour'] = df['created_date'].dt.hour

    df['engagement'] = df['upvote'] - df['downvote']
    df['total_votes'] = df['upvote'] + df['downvote']

# ===================== TF-IDF =====================
tfidf = TfidfVectorizer(
    max_features=18000,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9,
    stop_words='english'
)

X_text = tfidf.fit_transform(train['comment'])
X_test_text = tfidf.transform(test['comment'])

# ===================== NUMERIC =====================
drop_cols = ['comment', 'created_date', 'post_id']

train_num = train.drop(columns=drop_cols + ['label'])
test_num = test.drop(columns=drop_cols)

train_num = train_num.apply(pd.to_numeric, errors='coerce').fillna(0)
test_num = test_num.apply(pd.to_numeric, errors='coerce').fillna(0)

scaler = StandardScaler(with_mean=False)
train_scaled = scaler.fit_transform(train_num)
test_scaled = scaler.transform(test_num)

# ===================== COMBINE =====================
X = hstack([X_text, train_scaled])
X_test = hstack([X_test_text, test_scaled])

y = train['label']

# ===================== SPLIT =====================
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# ===================== MODELS =====================

# Logistic (Improved)
lr = LogisticRegression(max_iter=3000, solver='saga', C=0.7, n_jobs=-1)
lr.fit(X_train, y_train)
print("Logistic:", accuracy_score(y_val, lr.predict(X_val)))

# SGD
sgd = SGDClassifier(loss='log_loss', alpha=1e-4)
sgd.fit(X_train, y_train)
print("SGD:", accuracy_score(y_val, sgd.predict(X_val)))

# Naive Bayes (TEXT ONLY)
X_train_text, X_val_text, _, _ = train_test_split(X_text, y, test_size=0.2, random_state=42)

nb = MultinomialNB(alpha=0.5)
nb.fit(X_train_text, y_train)
print("Naive Bayes:", accuracy_score(y_val, nb.predict(X_val_text)))

# SVM (Improved)
svm = LinearSVC(C=0.5)
svm.fit(X_train, y_train)
print("SVM:", accuracy_score(y_val, svm.predict(X_val)))

# KNN
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train, y_train)
print("KNN:", accuracy_score(y_val, knn.predict(X_val)))

# ===================== BEST MODELS =====================

# LightGBM (TUNED + WARNINGS FIXED)
lgb = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=12,
    num_leaves=70,
    min_data_in_leaf=50,
    min_gain_to_split=0.01,
    verbose=-1
)
lgb.fit(X_train, y_train)
lgb_acc = accuracy_score(y_val, lgb.predict(X_val))
print("LightGBM:", lgb_acc)

# XGBoost (TUNED)
xgb = XGBClassifier(
    n_estimators=700,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    use_label_encoder=False
)
xgb.fit(X_train, y_train)
xgb_acc = accuracy_score(y_val, xgb.predict(X_val))
print("XGBoost:", xgb_acc)

# ===================== ENSEMBLE (BOOST SCORE) =====================
lgb_pred = lgb.predict(X_test)
xgb_pred = xgb.predict(X_test)

final_preds = (lgb_pred + xgb_pred) / 2
final_preds = np.round(final_preds).astype(int)

In [ ]:
# ===================== FINAL SUBMISSION (UPDATED TO SAMPLE FORMAT) =====================
sample_df = pd.read_csv('/kaggle/input/competitions/comment-category-prediction-challenge/Sample.csv')

print(sample_df.head())
print(len(sample_df))

# Use your already computed final predictions
test_predictions = final_preds

print(len(test_predictions), len(sample_df))

# Replace only label column
submission = sample_df.copy()
submission['label'] = test_predictions

# Save file
submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")